# Lab 09｜边界挡住一次操作后，任务怎样继续？

本练习准确范围是**预设配置边界观察和独立 Thread 的批准/拒绝对照**。这是两个独立观察：**目录写入范围**决定哪里能写；**客户端审批**决定某条具体命令这次是否获准。先看允许目录与受保护输入的区别，再看拒绝和允许两次运行。全部对象仅在本次教学目录内，不连接生产系统。

## 核心代码：在客户端核对具体动作，并消费一次性批准

> 本讲重点看下面这 10 行。它们摘自后面的代码单元格，仅供定位；运行时仍从公共准备开始按顺序执行。单元格从标题所在的第 1 格起计数，行号按格内源码计算。

**公共准备，第 3 个单元格的 `review()`，第 115 至 122 行：**

```python
exact = exact_command(raw, "python3 write_report.py")
matched = (method == "item/commandExecution/requestApproval" and exact
    and params.get("cwd") == str(outputs) and script.is_file() and not script.is_symlink()
    and hashlib.sha256(script.read_bytes()).hexdigest() == script_hash)
permitted = approved and matched
if permitted:
    approved = False  # 只批准本次具体动作，不留下长期通行状态。
decision = "accept" if permitted else "decline"
```

`exact_command()` 核对命令，`matched` 继续核对审批类型、目录和脚本哈希；只有匹配且已获批准才允许执行。`approved = False` 消费本次批准，其他情况返回拒绝决定。

**同一回调的返回行，第 125 行：**

```python
return {"decision": decision}
```

把决定返回给客户端；模型不能自行把拒绝改成允许。

**公共准备，第 3 个单元格的客户端创建处，第 127 行：**

```python
client = lab.manage_client(CodexClient(config=lab.config, approval_handler=review))
```

`approval_handler=review` 把回调接到实际客户端。第 2 节分别运行拒绝与允许的情形；目录写入边界另由第 1 节的 Sandbox 配置控制。


## 公共准备｜必跑，代码可折叠

选择 **Python 3.12** 内核，保留仓库的 `examples/notebook_support.py`。下格会自动补装固定版本的实验依赖；如果当前内核已经导入其他版本，请安装后重启内核。无需先到 README 手动安装。

九讲统一读取 `GLM_API_KEY`；未设置时弹出隐藏输入框。模型由 GLM-5.2 提供，不需要 OpenAI / ChatGPT 账号；开始任务后会产生 API 费用。准备区只启动本地连接，不调用模型。每讲可从本格独立启动，最后一格关闭连接；再次从头运行前先执行清理。

版本检查、协议适配、健康检查和退出处理属于辅助代码，不必逐行阅读。主流程以“必读必跑”标注；“扩展/可选”默认跳过。


In [ ]:
import sys, os, json, secrets, time, subprocess, getpass
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
from IPython.display import display, Markdown, HTML
from html import escape
from contextlib import redirect_stdout
from io import StringIO

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("请选择 Python 3.12 内核，再从头运行。")
required = {"openai-codex": "0.154.0", "litellm": "1.101.0"}
missing = []
for package, expected in required.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    if installed != expected:
        missing.append(("litellm[proxy]" if package == "litellm" else package) + "==" + expected)
if missing:
    already_loaded = any(name in sys.modules for name in ("openai_codex", "litellm"))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    if already_loaded:
        raise RuntimeError("依赖已安装；请重启内核并重新运行，避免混用旧模块。")

here = Path.cwd().resolve()
examples = next((p for base in (here, *here.parents)
                 for p in (base, base / "examples") if (p / "notebook_support.py").is_file()), None)
if examples is None:
    raise FileNotFoundError("请在完整仓库内打开本 notebook。")
sys.path.insert(0, str(examples))
from notebook_support import Lab, command_receipts, tool_output_contains
from openai_codex import ApprovalMode, Sandbox
if not os.environ.get("GLM_API_KEY", "").strip():
    os.environ["GLM_API_KEY"] = getpass.getpass("GLM_API_KEY：").strip()
if "lab" in globals() and not globals()["lab"]._closed:
    raise RuntimeError("本讲连接仍在运行，请先执行最后的 lab.close()。")
lab = Lab(examples / "09-isolation-approval")

# 展示函数只整理真实材料，不把预期结果冒充本次运行结果。
def table(headers, rows):
    head = "".join(f"<th>{escape(str(v))}</th>" for v in headers)
    body = "".join("<tr>" + "".join(
        "<td><pre style='white-space:pre-wrap;max-width:52em'>" + escape(lab.clean(v)) + "</pre></td>"
        for v in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

def details(title, value):
    display(HTML("<details><summary>" + escape(title) + "</summary><pre style='white-space:pre-wrap'>"
                 + escape(lab.clean(value)) + "</pre></details>"))

def run(thread, prompt, **kwargs):
    # 保留运行错误；仅收起辅助层重复打印的状态和回答。
    with redirect_stdout(StringIO()):
        return lab.run(thread, prompt, **kwargs)

def events(result):
    rows = []
    for wrapped in result.items:
        item = wrapped.root
        if item.type == "commandExecution":
            rows.extend([("采取的动作", item.command),
                         ("刚获得的信息", item.aggregated_output or "（无文本输出）")])
        elif item.type == "fileChange":
            rows.append(("文件操作", ", ".join(change.path for change in item.changes)))
        elif item.type == "agentMessage" and item.text.strip():
            rows.append(("Agent 的判断", item.text))
    table(["事件", "本次实际内容"], rows)

def snapshot(folder):
    return {str(p.relative_to(folder)):p.read_bytes() for p in folder.rglob("*") if p.is_file()}

def compare_inputs(folder, before):
    after = snapshot(folder)
    changed = [name for name in sorted(before.keys() | after.keys()) if before.get(name) != after.get(name)]
    print("输入文件：" + ("有变化，展开下方查看" if changed else "未改变"))
    for name in changed:
        details(name, "运行前：\n" + before.get(name,"[不存在]".encode()).decode(errors="replace")
                + "\n运行后：\n" + after.get(name,"[不存在]".encode()).decode(errors="replace"))



WORK = lab.work
import hashlib, shlex
from notebook_support import denied_write, exact_command
inputs = WORK / "inputs"
outputs = WORK / "outputs"
inputs.mkdir()
outputs.mkdir()
protected = inputs / "source.txt"
protected.write_text("02:07 支付请求延迟升高；连接等待 3600ms，CPU 24%。原因仍待核对。",encoding="utf-8")
input_hash = hashlib.sha256(protected.read_bytes()).hexdigest()
probe_command = f"printf changed > {shlex.quote(str(protected.resolve()))}"

def sandbox_observation(receipts,command):
    output = "\n".join(r["output"] for r in receipts)
    if "sandbox_apply" in output:
        display(Markdown("**本次无法观察权限边界**：宿主环境阻止了嵌套 Sandbox 启动；这不是对目标文件的权限拒绝。请在可启动原生 Sandbox 的本机环境重跑。"))
        return "Sandbox 未启动"
    return "实际写入被拒绝" if denied_write(receipts,command,protected) else "未取得匹配的权限拒绝回执"
import shlex
from openai_codex.client import CodexClient
from openai_codex.api import Thread

script = outputs / "write_report.py"
script.write_text("from pathlib import Path\nPath('report.txt').write_text('已批准的教学报告', encoding='utf-8')\n", encoding="utf-8")
script_hash = hashlib.sha256(script.read_bytes()).hexdigest()
approved = False
approvals = []

def review(method, params):
    params = params or {}
    global approved
    raw = params.get("command", "")
    exact = exact_command(raw, "python3 write_report.py")
    matched = (method == "item/commandExecution/requestApproval" and exact
        and params.get("cwd") == str(outputs) and script.is_file() and not script.is_symlink()
        and hashlib.sha256(script.read_bytes()).hexdigest() == script_hash)
    permitted = approved and matched
    if permitted:
        approved = False  # 只批准本次具体动作，不留下长期通行状态。
    decision = "accept" if permitted else "decline"
    approvals.append({"method": method, "command": lab.clean(raw), "cwd": lab.clean(params.get("cwd")),
                      "matched": matched, "decision": decision})
    return {"decision": decision}

client = lab.manage_client(CodexClient(config=lab.config, approval_handler=review))
client.start()
client.initialize()

def approval_thread():
    started = client.thread_start({"cwd": str(outputs), "sandbox": "workspace-write",
        "approvalPolicy": "untrusted", "approvalsReviewer": "user",
        "developerInstructions": "只执行用户给出的那条命令；若审批拒绝就停止，不换其他工具或命令。"})
    return Thread(client, started.thread.id)
print("公共准备完成。")


## 1. 写入越界被拒后，在允许范围完成报告｜必读必跑

先做一次很短的全只读探针，再把 `outputs/` 设为可写工作目录。Agent 运行一次边界探针，能写允许文件，却不能改相邻 `inputs/`。随后继续读取材料、在允许目录生成分析报告，不把“拒绝”当成整个目标的终点。

本节统一使用 Python SDK：`thread_start(cwd=..., sandbox=..., approval_mode=...)` 配置会话，`lab.run()` 经 `Thread.turn()` 启动并等待任务。完整调用在下格；不把 `turn/start.params` 的 `sandboxPolicy` 与 CLI/TOML 字段拼在一起。输入文件和输出目录已由公共准备创建。

可写会话的 `cwd` 明确设为 `outputs/`，相邻 `inputs/` 不在该可写目录中；临时目录例外及网络配置由本讲共用的 `lab.config` 预设。本例观察预设边界，未从自然语言计划自动生成或应用权限。


In [ ]:
probe_start = len(lab.requests)
readonly = lab.codex.thread_start(
    cwd=str(WORK), sandbox=Sandbox.read_only, approval_mode=ApprovalMode.deny_all,
)
probe_result = run(readonly, f"原样执行一次教学探针：`{probe_command}`。记录结果即可，不重试、不申请扩权。")
probe_receipts = command_receipts(lab.requests[probe_start:])
readonly_state = sandbox_observation(probe_receipts,probe_command)
print("全只读前置观察：",readonly_state)

allowed = outputs / "allowed.txt"
scoped_command = f"printf allowed > {shlex.quote(str(allowed))} && {probe_command}"
scoped = lab.codex.thread_start(
    cwd=str(outputs), sandbox=Sandbox.workspace_write, approval_mode=ApprovalMode.deny_all,
)
scoped_start = len(lab.requests)
scoped_result = run(scoped,
    f"目标是根据 {protected} 完成一份简短支付告警分析，写到当前目录 incident-report.md。"
    f"先原样执行一次教学边界探针：`{scoped_command}`。"
    "如果修改输入被拒绝，保留原文件，不重试写入、不申请扩权；继续只读分析，在当前允许目录完成报告。")
scoped_receipts = command_receipts(lab.requests[scoped_start:])
scoped_state = sandbox_observation(scoped_receipts,scoped_command)
report = outputs / "incident-report.md"
table(["动作", "实际结果"], [
    ("全只读目录的写入",readonly_state),
    ("允许目录 outputs/ 写入",allowed.read_text() if allowed.exists() else "未生成允许文件"),
    ("越界修改 inputs/",scoped_state),
    ("受保护输入", "未改变" if hashlib.sha256(protected.read_bytes()).hexdigest()==input_hash else "已改变"),
    ("拒绝后的分析报告",report.read_text() if report.exists() else "未生成，回看事件判断停在何处"),
])
events(scoped_result)


## 2. 对具体命令作客户端审批｜必读必跑

底层客户端配置 `approvalPolicy=untrusted`、`approvalsReviewer=user`。决定来自 notebook 的审批回调，模型不能自行批准。回调识别固定目录、`python3 write_report.py` 和脚本哈希，其他请求一律拒绝；解析实现留在公共准备区。

下格是**两次新 Thread 的独立运行**：第一次拒绝，第二次允许同一条教学命令。不是同一任务中先拒绝再批准，也不是生产系统的授权界面。


In [ ]:
task = "请只运行 python3 write_report.py，不读取或改写脚本。若审批拒绝立即停止；成功后报告状态。"
(outputs / "report.txt").unlink(missing_ok=True)  # 仅清除本次目录中的演示报告。
denied_start = len(approvals)
denied_thread = approval_thread()
denied = run(denied_thread,task)
denied_events = approvals[denied_start:]
report_after_denial = (outputs / "report.txt").exists()
approved = True
accepted_start = len(approvals)
accepted_request_start = len(lab.requests)
accepted_thread = approval_thread()
try:
    accepted = run(accepted_thread,task)
finally:
    approval_consumed = not approved
    approved = False
accepted_events = approvals[accepted_start:]
accepted_receipts = command_receipts(lab.requests[accepted_request_start:])
table(["独立运行", "客户端实际决定", "文件结果"], [
    ("新 Thread 1 / 拒绝", ", ".join(e["decision"] for e in denied_events) or "未收到审批请求",
     "报告存在" if report_after_denial else "未生成报告"),
    ("新 Thread 2 / 允许", ", ".join(e["decision"] for e in accepted_events) or "未收到审批请求",
     (outputs / "report.txt").read_text() if (outputs / "report.txt").exists() else "未生成报告"),
])
details("扩展：具体命令、哈希匹配、回执与一次性批准",json.dumps({
    "审批":approvals, "批准是否被匹配命令消费":approval_consumed,
    "允许后的工具回执":accepted_receipts,
    "两个 Thread 是否不同":denied_thread.id != accepted_thread.id,
    "脚本是否仍为原内容":hashlib.sha256(script.read_bytes()).hexdigest()==script_hash,
},ensure_ascii=False,indent=2))


## 继续观察｜扩展/可选

批准以后脚本内容变了，同一条命令还应该继续获准吗？当前回调在审批时核对目录、命令和脚本哈希；检查与执行之间仍有时间间隙，不能据此认为已防住并发替换。批准也不证明报告正确。

下格可生成一份**权限配置建议草案**，列出输入只读、报告目录可写和需要逐次审批的动作。仅用于人工核对，不自动应用；若需要实际改变权限配置，应在核对后另行操作。本例没有单独验证网络边界。


In [ ]:
RUN_PERMISSION_PLAN = False
if RUN_PERMISSION_PLAN:
    plan = run(lab.thread(),
        "为本教学项目提出一份权限配置建议：inputs 保存原始材料，outputs 保存报告。"
        "说明哪些动作需逐次审批，并列出应用前应由人核对的事项。只输出草案，不调用工具或应用配置。")
    display(Markdown(plan.final_response))


In [ ]:
# 继续观察结束后关闭本讲全部客户端与适配器。
lab.close()
